# 10 - LangGraph y Flujos de Trabajo

## Curso de LLMs y Aplicaciones de IA

**Duración estimada:** 2.5-3 horas

---

## Índice

1. [Introducción a LangGraph](#intro)
2. [Estados y Grafos](#estados)
3. [Flujo RAG con LangGraph](#rag)
4. [Auto-corrección](#correccion)
5. [Checkpoints y Persistencia](#checkpoints)
6. [Ejercicios prácticos](#ejercicios)

---

## Objetivos de aprendizaje

Al finalizar este notebook, serás capaz de:
- Crear grafos de estados con LangGraph
- Implementar flujos condicionales
- Añadir auto-corrección a sistemas RAG
- Usar checkpoints para persistencia

<a name="intro"></a>
## 1. Introducción a LangGraph

**LangGraph** es una librería de LangChain para crear flujos de trabajo como grafos de estados.

### ¿Por qué LangGraph?

- **Control explícito**: Define exactamente el flujo
- **Condicionales**: Diferentes caminos según resultados
- **Ciclos**: Permite iteraciones y re-intentos
- **Estado**: Mantiene información entre nodos
- **Persistencia**: Checkpoints para recuperación

In [2]:
# Install
#!pip install -q langchain langchain-groq langgraph langchain-huggingface faiss-cpu

In [4]:
import os
from getpass import getpass
import warnings
warnings.filterwarnings('ignore')

if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass("GROQ API Key: ")

from langchain_groq import ChatGroq
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)
print("Configurado ✓")

Configurado ✓


<a name="estados"></a>
## 2. Estados y Grafos

En LangGraph, definimos:
- **State**: Datos que fluyen por el grafo
- **Nodes**: Funciones que procesan el estado
- **Edges**: Conexiones entre nodos

In [5]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

# Define state
class SimpleState(TypedDict):
    messages: List[str]
    current_step: str

# Define nodes
def step_one(state: SimpleState) -> SimpleState:
    messages = state["messages"] + ["Paso 1 completado"]
    return {"messages": messages, "current_step": "one"}

def step_two(state: SimpleState) -> SimpleState:
    messages = state["messages"] + ["Paso 2 completado"]
    return {"messages": messages, "current_step": "two"}

def step_three(state: SimpleState) -> SimpleState:
    messages = state["messages"] + ["Paso 3 completado"]
    return {"messages": messages, "current_step": "three"}

# Build graph
workflow = StateGraph(SimpleState)
workflow.add_node("step_one", step_one)
workflow.add_node("step_two", step_two)
workflow.add_node("step_three", step_three)

# Add edges
workflow.add_edge(START, "step_one")
workflow.add_edge("step_one", "step_two")
workflow.add_edge("step_two", "step_three")
workflow.add_edge("step_three", END)

# Compile
app = workflow.compile()
print("Grafo compilado ✓")

Grafo compilado ✓


In [6]:
# Run the graph
result = app.invoke({"messages": ["Inicio"], "current_step": ""})

print("Resultado:")
for msg in result["messages"]:
    print(f"  - {msg}")

Resultado:
  - Inicio
  - Paso 1 completado
  - Paso 2 completado
  - Paso 3 completado


### Grafos con condicionales

In [7]:
from typing import Literal

class ConditionalState(TypedDict):
    query: str
    query_type: str
    response: str

def classify_query(state: ConditionalState) -> ConditionalState:
    """Classify the query type."""
    query = state["query"].lower()
    if "precio" in query or "costo" in query:
        return {**state, "query_type": "pricing"}
    elif "horario" in query or "hora" in query:
        return {**state, "query_type": "schedule"}
    else:
        return {**state, "query_type": "general"}

def handle_pricing(state: ConditionalState) -> ConditionalState:
    return {**state, "response": "Los precios son: Básico 99€, Pro 299€, Enterprise consultar."}

def handle_schedule(state: ConditionalState) -> ConditionalState:
    return {**state, "response": "Horario: Lunes a Viernes, 9:00 a 18:00."}

def handle_general(state: ConditionalState) -> ConditionalState:
    return {**state, "response": "Para más información, contacta con soporte@empresa.com"}

def route_query(state: ConditionalState) -> Literal["pricing", "schedule", "general"]:
    return state["query_type"]

# Build conditional graph
cond_workflow = StateGraph(ConditionalState)
cond_workflow.add_node("classify", classify_query)
cond_workflow.add_node("pricing", handle_pricing)
cond_workflow.add_node("schedule", handle_schedule)
cond_workflow.add_node("general", handle_general)

cond_workflow.add_edge(START, "classify")
cond_workflow.add_conditional_edges(
    "classify",
    route_query,
    {"pricing": "pricing", "schedule": "schedule", "general": "general"}
)
cond_workflow.add_edge("pricing", END)
cond_workflow.add_edge("schedule", END)
cond_workflow.add_edge("general", END)

cond_app = cond_workflow.compile()
print("Grafo condicional compilado ✓")

Grafo condicional compilado ✓


In [8]:
# Test conditional routing
queries = [
    "¿Cuál es el precio del plan básico?",
    "¿Cuál es el horario de atención?",
    "¿Tienen servicio en México?"
]

for q in queries:
    result = cond_app.invoke({"query": q, "query_type": "", "response": ""})
    print(f"Q: {q}")
    print(f"A: {result['response']}\n")

Q: ¿Cuál es el precio del plan básico?
A: Los precios son: Básico 99€, Pro 299€, Enterprise consultar.

Q: ¿Cuál es el horario de atención?
A: Horario: Lunes a Viernes, 9:00 a 18:00.

Q: ¿Tienen servicio en México?
A: Para más información, contacta con soporte@empresa.com



<a name="rag"></a>
## 3. Flujo RAG con LangGraph

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage

# Create vector store
docs = [
    Document(page_content="El IBI se paga anualmente basado en el valor catastral."),
    Document(page_content="El IVTM grava la titularidad de vehículos matriculados."),
    Document(page_content="El ICIO se liquida al finalizar construcciones u obras."),
    Document(page_content="Las bonificaciones pueden reducir hasta un 90% el impuesto."),
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("Vector store creado ✓")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store creado ✓


In [10]:
from typing import List
from langchain_core.messages import BaseMessage

class RAGState(TypedDict):
    messages: List[BaseMessage]
    context: str
    response: str

def retrieve_context(state: RAGState) -> RAGState:
    """Retrieve relevant documents."""
    query = state["messages"][-1].content
    docs = retriever.invoke(query)
    context = "\n".join([d.page_content for d in docs])
    return {**state, "context": context}

def generate_response(state: RAGState) -> RAGState:
    """Generate response using LLM."""
    query = state["messages"][-1].content
    context = state["context"]
    
    prompt = f"""Responde basándote en el contexto.
    
Contexto: {context}

Pregunta: {query}

Respuesta:"""
    
    response = llm.invoke(prompt)
    return {**state, "response": response.content}

# Build RAG graph
rag_workflow = StateGraph(RAGState)
rag_workflow.add_node("retrieve", retrieve_context)
rag_workflow.add_node("generate", generate_response)

rag_workflow.add_edge(START, "retrieve")
rag_workflow.add_edge("retrieve", "generate")
rag_workflow.add_edge("generate", END)

rag_app = rag_workflow.compile()
print("RAG graph compilado ✓")

RAG graph compilado ✓


In [11]:
# Test RAG
result = rag_app.invoke({
    "messages": [HumanMessage(content="¿Qué es el IBI?")],
    "context": "",
    "response": ""
})

print(f"Respuesta: {result['response']}")

Respuesta: El IBI (Impuesto de Bienes Inmuebles) es un impuesto que se paga anualmente y se basa en el valor catastral de un inmueble. Se aplica a los propietarios de bienes inmuebles, como casas, apartamentos, terrenos, etc. El objetivo del IBI es recaudar fondos para las administraciones locales, que luego se utilizan para financiar servicios y infraestructuras públicas en la comunidad.


<a name="correccion"></a>
## 4. Auto-corrección

Añadimos un paso de verificación y corrección.

In [12]:
class CorrectionState(TypedDict):
    query: str
    context: str
    response: str
    corrected_response: str
    needs_correction: bool

def retrieve(state: CorrectionState) -> CorrectionState:
    docs = retriever.invoke(state["query"])
    context = "\n".join([d.page_content for d in docs])
    return {**state, "context": context}

def generate(state: CorrectionState) -> CorrectionState:
    prompt = f"Contexto: {state['context']}\nPregunta: {state['query']}\nRespuesta:"
    response = llm.invoke(prompt)
    return {**state, "response": response.content}

def check_response(state: CorrectionState) -> CorrectionState:
    """Check if response needs correction."""
    check_prompt = f"""¿La siguiente respuesta está basada en el contexto?
    
Contexto: {state['context']}
Respuesta: {state['response']}

Responde solo 'SI' o 'NO'."""
    
    check = llm.invoke(check_prompt)
    needs_correction = "NO" in check.content.upper()
    return {**state, "needs_correction": needs_correction}

def correct_response(state: CorrectionState) -> CorrectionState:
    """Correct the response."""
    correct_prompt = f"""Mejora esta respuesta basándote solo en el contexto.
    
Contexto: {state['context']}
Respuesta original: {state['response']}

Respuesta mejorada:"""
    
    corrected = llm.invoke(correct_prompt)
    return {**state, "corrected_response": corrected.content}

def route_correction(state: CorrectionState) -> Literal["correct", "end"]:
    return "correct" if state["needs_correction"] else "end"

# Build correction graph
corr_workflow = StateGraph(CorrectionState)
corr_workflow.add_node("retrieve", retrieve)
corr_workflow.add_node("generate", generate)
corr_workflow.add_node("check", check_response)
corr_workflow.add_node("correct", correct_response)

corr_workflow.add_edge(START, "retrieve")
corr_workflow.add_edge("retrieve", "generate")
corr_workflow.add_edge("generate", "check")
corr_workflow.add_conditional_edges("check", route_correction, {"correct": "correct", "end": END})
corr_workflow.add_edge("correct", END)

corr_app = corr_workflow.compile()
print("Grafo con corrección compilado ✓")

Grafo con corrección compilado ✓


In [13]:
# Test
result = corr_app.invoke({
    "query": "¿Cuándo se paga el IVTM?",
    "context": "",
    "response": "",
    "corrected_response": "",
    "needs_correction": False
})

print(f"Respuesta original: {result['response']}")
print(f"Necesitó corrección: {result['needs_correction']}")
if result['corrected_response']:
    print(f"Respuesta corregida: {result['corrected_response']}")

Respuesta original: El IVTM se paga anualmente, al igual que el IBI, pero en este caso, se grava la titularidad de vehículos matriculados. Por lo tanto, la respuesta es: anualmente.
Necesitó corrección: False


<a name="checkpoints"></a>
## 5. Checkpoints y Persistencia

In [14]:
from langgraph.checkpoint.memory import MemorySaver

# Create checkpointer
memory = MemorySaver()

# Compile with checkpointer
rag_with_memory = rag_workflow.compile(checkpointer=memory)

# Run with thread_id for session tracking
config = {"configurable": {"thread_id": "session1"}}

result = rag_with_memory.invoke({
    "messages": [HumanMessage(content="¿Qué impuestos hay?")],
    "context": "",
    "response": ""
}, config=config)

print(f"Respuesta: {result['response']}")

Respuesta: Hay varios impuestos, pero en el contexto proporcionado, se mencionan dos:

1. **Impuesto sobre la renta o beneficios**: Se menciona que las bonificaciones pueden reducir hasta un 90% este impuesto, aunque no se especifica su nombre exacto.
2. **Impuesto sobre Bienes Inmuebles (IBI)**: Este impuesto se paga anualmente y está basado en el valor catastral de la propiedad.


<a name="ejercicios"></a>
## 6. Ejercicios Prácticos

### Ejercicio: Crear un flujo con múltiples pasos

In [15]:
# Exercise: Create a workflow that:
# 1. Receives a question
# 2. Classifies the question type
# 3. Retrieves relevant info
# 4. Generates response
# 5. Checks quality
# 6. Corrects if needed

In [17]:
from typing import TypedDict, Literal


class FullWorkflowState(TypedDict):
    question: str
    question_type: str
    context: str
    response: str
    quality_passed: bool
    quality_feedback: str
    final_response: str


# 1 + 2. Recibe la pregunta y clasifica el tipo
def classify_question(state: FullWorkflowState) -> FullWorkflowState:
    question = state["question"].lower()

    if any(word in question for word in ["ibi", "ivtm", "icio", "impuesto", "bonificación", "bonificaciones"]):
        question_type = "tax_question"
    elif any(word in question for word in ["cuándo", "cuando", "paga", "liquida", "finalizar"]):
        question_type = "procedure_question"
    else:
        question_type = "general_question"

    return {
        **state,
        "question_type": question_type
    }


# 3. Recupera información relevante
def retrieve_relevant_info(state: FullWorkflowState) -> FullWorkflowState:
    docs = retriever.invoke(state["question"])
    context = "\n".join([doc.page_content for doc in docs])

    return {
        **state,
        "context": context
    }


# 4. Genera respuesta
def generate_answer(state: FullWorkflowState) -> FullWorkflowState:
    prompt = f"""
Eres un asistente especializado en responder preguntas usando únicamente el contexto proporcionado.

Tipo de pregunta: {state["question_type"]}

Contexto:
{state["context"]}

Pregunta:
{state["question"]}

Instrucciones:
- Responde de forma clara y breve.
- Usa solo la información del contexto.
- Si el contexto no contiene información suficiente, indícalo.

Respuesta:
"""

    response = llm.invoke(prompt)

    return {
        **state,
        "response": response.content
    }


# 5. Comprueba calidad
def check_quality(state: FullWorkflowState) -> FullWorkflowState:
    check_prompt = f"""
Evalúa si la respuesta es válida.

Contexto:
{state["context"]}

Pregunta:
{state["question"]}

Respuesta:
{state["response"]}

Criterios:
1. La respuesta debe estar basada en el contexto.
2. No debe inventar información.
3. Debe responder a la pregunta.
4. Debe ser clara.

Responde únicamente con este formato:
PASS: explicación breve
o
FAIL: explicación breve
"""

    check = llm.invoke(check_prompt)
    feedback = check.content.strip()

    quality_passed = feedback.upper().startswith("PASS")

    final_response = state["response"] if quality_passed else ""

    return {
        **state,
        "quality_passed": quality_passed,
        "quality_feedback": feedback,
        "final_response": final_response
    }


# 6. Corrige si es necesario
def correct_if_needed(state: FullWorkflowState) -> FullWorkflowState:
    correction_prompt = f"""
La respuesta anterior no ha superado el control de calidad.

Contexto:
{state["context"]}

Pregunta:
{state["question"]}

Respuesta original:
{state["response"]}

Feedback de calidad:
{state["quality_feedback"]}

Genera una nueva respuesta corregida:
- Basada solo en el contexto.
- Sin inventar información.
- Clara y directa.

Respuesta corregida:
"""

    corrected = llm.invoke(correction_prompt)

    return {
        **state,
        "final_response": corrected.content
    }


# Router condicional después del quality check
def route_after_quality_check(state: FullWorkflowState) -> Literal["correct", "end"]:
    if state["quality_passed"]:
        return "end"
    else:
        return "correct"


# Construcción del grafo completo
full_workflow = StateGraph(FullWorkflowState)

full_workflow.add_node("classify_question", classify_question)
full_workflow.add_node("retrieve_relevant_info", retrieve_relevant_info)
full_workflow.add_node("generate_answer", generate_answer)
full_workflow.add_node("check_quality", check_quality)
full_workflow.add_node("correct_if_needed", correct_if_needed)

full_workflow.add_edge(START, "classify_question")
full_workflow.add_edge("classify_question", "retrieve_relevant_info")
full_workflow.add_edge("retrieve_relevant_info", "generate_answer")
full_workflow.add_edge("generate_answer", "check_quality")

full_workflow.add_conditional_edges(
    "check_quality",
    route_after_quality_check,
    {
        "correct": "correct_if_needed",
        "end": END
    }
)

full_workflow.add_edge("correct_if_needed", END)

full_app = full_workflow.compile()

print("Workflow completo compilado correctamente ✓")

Workflow completo compilado correctamente ✓


In [18]:
test_questions = [
    "¿Qué es el IBI?",
    "¿Cuándo se paga el IVTM?",
    "¿Qué bonificaciones existen?",
    "¿Qué es el impuesto de sociedades?"
]

for question in test_questions:
    result = full_app.invoke({
        "question": question,
        "question_type": "",
        "context": "",
        "response": "",
        "quality_passed": False,
        "quality_feedback": "",
        "final_response": ""
    })

    print("=" * 80)
    print(f"Pregunta: {question}")
    print(f"Tipo clasificado: {result['question_type']}")
    print(f"\nContexto recuperado:\n{result['context']}")
    print(f"\nRespuesta inicial:\n{result['response']}")
    print(f"\nQuality passed: {result['quality_passed']}")
    print(f"Quality feedback: {result['quality_feedback']}")
    print(f"\nRespuesta final:\n{result['final_response']}")

Pregunta: ¿Qué es el IBI?
Tipo clasificado: tax_question

Contexto recuperado:
El IBI se paga anualmente basado en el valor catastral.
El ICIO se liquida al finalizar construcciones u obras.

Respuesta inicial:
El IBI es un impuesto que se paga anualmente y se basa en el valor catastral.

Quality passed: True
Quality feedback: PASS: La respuesta se basa en el contexto proporcionado y responde directamente a la pregunta, sin inventar información adicional y de manera clara.

Respuesta final:
El IBI es un impuesto que se paga anualmente y se basa en el valor catastral.
Pregunta: ¿Cuándo se paga el IVTM?
Tipo clasificado: tax_question

Contexto recuperado:
El IBI se paga anualmente basado en el valor catastral.
El IVTM grava la titularidad de vehículos matriculados.

Respuesta inicial:
El contexto no contiene información suficiente sobre la frecuencia de pago del IVTM. Solo se menciona que grava la titularidad de vehículos matriculados, pero no se especifica cuándo se paga.

Quality passe

## Resumen

En este notebook hemos aprendido:

1. **LangGraph**: Crear flujos como grafos de estados
2. **Condicionales**: Routing basado en resultados
3. **RAG workflow**: Retrieve → Generate
4. **Auto-corrección**: Verificar y mejorar respuestas
5. **Checkpoints**: Persistencia de sesiones

En el siguiente notebook veremos **RAG Avanzado Agentic** con flujos completos.

---

## Referencias

- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)

In [16]:
import session_info
session_info.show(html=False)

-----
ipykernel                   6.31.0
langchain_community         0.4.2
langchain_core              1.4.0
langchain_groq              1.1.2
langchain_huggingface       NA
langgraph                   NA
pandas                      2.3.3
session_info                v1.0.1
-----
IPython             9.7.0
jupyter_client      8.6.3
jupyter_core        5.8.1
jupyterlab          4.4.7
notebook            7.4.5
-----
Python 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Windows-10-10.0.19045-SP0
-----
Session information updated at 2026-06-17 18:53
